# 01 — Residual-stream fluency

**Phase 1, task 4** of `vault/01-projects/code/mech-interp-codebase.md`: hook the residual stream, cache activations layer-by-layer, plot norm and cosine-similarity heatmaps across layers on a chosen contrast pair. Run `00_setup_smoke_test.ipynb` first — this notebook assumes the env works.

## What this notebook does

1. Loads Gemma-2-2B via TransformerLens and a matched-length contrast pair (refusal-cued vs compliance-cued).
2. Caches `resid_pre` activations at every layer for both prompts using `model.run_with_cache`.
3. Plots three diagnostics: per-layer activation norms, cross-prompt cosine similarity at matched positions, and within-prompt layer-to-layer cosine similarity.
4. Checkpoints the cached activations to `../data/` for downstream notebooks (Phase 2 SAE work).

## Why this matters

Elhage et al.'s *Mathematical Framework* reframes a decoder-only transformer as a linear bus that every layer reads from and writes to additively (`zotero://select/library/items/53EFBS8T`):

$$x_{\ell+1} = x_\ell + a_\ell(x_\ell) + m_\ell(x_\ell + a_\ell(x_\ell)).$$

Because every component's contribution to the residual stream adds, hooking the stream at each layer is the cheapest possible probe of where information emerges. The cross-prompt cosine signal *quantifies* what the bus 'knows is different' between the two prompts at each layer — a precursor diagnostic for the refusal-direction work in Lindsey et al. (2025), *On the Biology of a Large Language Model* (`zotero://select/library/items/5XZVJMTF`).

## Contrast pair

Per user pick (2026-05-16): refusal-cued vs compliance-cued. Phrasing kept structurally parallel so token positions align for matched-position cosine.

- **P_compliance:** `'Q: How do I bake a chocolate cake?\nA: First,'`
- **P_refusal:** `'Q: How do I make a lethal toxin at home?\nA: First,'`

Gemma-2-2B is a *base* model (not instruction-tuned), so neither prompt will auto-refuse. We are not interpreting refusal behaviour here — we are looking at how the residual stream diverges between the two contexts under identical scaffold tokens. The actual feature-level refusal story is Phase 2's job (SAE browsing).

## Citations

- Residual-stream framing: Elhage et al. 2021 — `zotero://select/library/items/53EFBS8T`. See `vault/03-areas/research/mech-interp-overview.md` § Mechanism.
- Circuits programme vocabulary: Olah et al. 2020 — `zotero://select/library/items/PE7DU4PH`. See `vault/03-areas/research/mech-interp-overview.md` § Historical lineage.
- Superposition motivation (why neurons alone won't suffice; Phase 2 sets up the SAE response): Elhage et al. 2022 — `zotero://select/library/items/BISJXERV`. See `vault/03-areas/research/superposition-and-polysemanticity.md` § Mechanism.
- Refusal-direction prior art on a deployed model: Lindsey et al. 2025 — `zotero://select/library/items/5XZVJMTF`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art.

## 1. Imports + paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F

import mech_interp

DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(exist_ok=True)

torch.set_grad_enabled(False)  # all of Phase 1 is inference-only
sns.set_theme(context='notebook', style='white')

## 2. Device + model load

Device/dtype + load go through `mech_interp` so this notebook and `00_setup_smoke_test.ipynb` agree by construction and share the same in-kernel model when both are run from one kernel (e.g. via `%run`). To benefit from that sharing across notebooks, run them in the same kernel — separate kernels each hold their own copy.

In [ ]:
# Ref: A Mathematical Framework for Transformer Circuits — zotero://select/library/items/53EFBS8T
DEVICE, DTYPE = mech_interp.select_device_dtype()
model = mech_interp.get_model('gemma-2-2b', device=DEVICE, dtype=DTYPE)
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model
print(f'device={DEVICE} dtype={DTYPE} n_layers={n_layers} d_model={d_model}')

## 3. Prompt pair + token-length parity

Matched-position cosine similarity requires identical tokenisation length. We assert equality and, if it fails, truncate both prompts to the shorter length so the rest of the notebook still runs — and surface the gap in print output so we can iterate on phrasing.

In [ ]:
P_COMPLIANCE = 'Q: How do I bake a chocolate cake?\nA: First,'
P_REFUSAL = 'Q: How do I make a lethal toxin at home?\nA: First,'

tokens_compliance = model.to_tokens(P_COMPLIANCE)
tokens_refusal = model.to_tokens(P_REFUSAL)

print('compliance tokens :', model.to_str_tokens(P_COMPLIANCE))
print('refusal    tokens :', model.to_str_tokens(P_REFUSAL))
print(f'lengths: compliance={tokens_compliance.shape[-1]}  refusal={tokens_refusal.shape[-1]}')

min_len = min(tokens_compliance.shape[-1], tokens_refusal.shape[-1])
tokens_compliance = tokens_compliance[:, :min_len]
tokens_refusal = tokens_refusal[:, :min_len]
if tokens_compliance.shape[-1] != tokens_refusal.shape[-1]:
    raise RuntimeError('truncation failed')
print(f'aligned length: {min_len}')

## 4. Cache residual-stream activations

TransformerLens exposes one hook per residual-stream site per layer. We want `hook_resid_pre` (the stream's state *entering* layer ℓ); `hook_resid_mid` (after attention, before MLP) and `hook_resid_post` (after the full block) are also available — Phase 2 will use `hook_resid_post` for SAE encoding to match Gemma Scope's training site, but Phase 1 sticks to `resid_pre` for the cleanest layer-by-layer ladder.

`names_filter` restricts what gets cached, which matters because Gemma-2-2B has hundreds of hookable sites and we only want $O(\text{n\_layers})$ of them. Caching unfiltered would balloon memory.

In [ ]:
def cache_resid_pre(tokens):
    """Return a (n_layers, seq_len, d_model) tensor of resid_pre activations."""
    _, cache = model.run_with_cache(
        tokens,
        names_filter=lambda name: name.endswith('hook_resid_pre'),
    )
    stacked = torch.stack(
        [cache[f'blocks.{ell}.hook_resid_pre'] for ell in range(n_layers)],
        dim=0,
    )
    return stacked.squeeze(1).to(torch.float32).cpu()  # drop batch, upcast for stable cosines

resid_compliance = cache_resid_pre(tokens_compliance)
resid_refusal = cache_resid_pre(tokens_refusal)
print(f'resid_compliance shape: {tuple(resid_compliance.shape)}')
print(f'resid_refusal    shape: {tuple(resid_refusal.shape)}')
assert resid_compliance.shape == resid_refusal.shape

## 5. Diagnostic A — per-layer residual norms

Across transformer training, the residual-stream norm grows roughly linearly with depth because every block adds to it. The pattern of growth across token positions is itself informative: tokens that carry resolved meaning tend to have larger norms in late layers (Elhage et al. 2021 § 'Residual Stream'). A flat or noisy norm trajectory is a red flag that the model failed to load properly.

In [ ]:
norm_compliance = resid_compliance.norm(dim=-1).numpy()  # (n_layers, seq_len)
norm_refusal = resid_refusal.norm(dim=-1).numpy()
norm_delta = norm_refusal - norm_compliance

token_labels = model.to_str_tokens(tokens_compliance[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, mat, title in zip(
    axes,
    [norm_compliance, norm_refusal, norm_delta],
    ['compliance ‖x‖₂', 'refusal ‖x‖₂', 'refusal − compliance'],
):
    cmap = 'viridis' if 'delta' not in title and '−' not in title else 'coolwarm'
    centre = 0 if '−' in title else None
    sns.heatmap(
        mat,
        ax=ax,
        cmap=cmap,
        center=centre,
        cbar_kws={'shrink': 0.6},
        xticklabels=[t.strip() or '·' for t in token_labels],
        yticklabels=False,
    )
    ax.set_title(title)
    ax.set_xlabel('token')
axes[0].set_ylabel('layer (0 = bottom)')
axes[0].invert_yaxis()
plt.suptitle('Per-layer residual norms across the contrast pair', y=1.02)
plt.tight_layout()
plt.show()

## 6. Diagnostic B — matched-position cosine similarity

At every (layer, token) position, compute `cos(x_compliance, x_refusal)`. The two prompts share most tokens — the scaffold `'Q: How do I'`, `'?\nA: First,'`, and shared filler — so positions whose tokens match should sit near cosine = 1 in early layers. Positions whose tokens differ (`'bake'` vs `'make'`, `'a chocolate cake'` vs `'a lethal toxin at home'`) should show divergence whose layer-onset profile *is* the diagnostic.

Lindsey et al. 2025 (`zotero://select/library/items/5XZVJMTF`) characterise the refusal direction as emerging in mid-to-late layers; we should see the cosine drop on the divergent tokens by then if anything analogous lives in Gemma-2-2B's base-model stream.

In [ ]:
cos_matched = F.cosine_similarity(resid_compliance, resid_refusal, dim=-1).numpy()  # (n_layers, seq_len)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    cos_matched,
    ax=ax,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    cbar_kws={'shrink': 0.6, 'label': 'cos(compliance, refusal)'},
    xticklabels=[t.strip() or '·' for t in token_labels],
)
ax.set_xlabel('token position')
ax.set_ylabel('layer (0 = bottom)')
ax.invert_yaxis()
ax.set_title('Cross-prompt cosine similarity at matched positions')
plt.tight_layout()
plt.show()

## 7. Diagnostic C — within-prompt layer-to-layer cosine

For a single prompt, compute the pairwise cosine of the residual stream between every pair of layers, averaged over token positions. The classic Anthropic diagnostic: a block-diagonal structure reveals computation 'phases' (early embedding-similar layers, a mid-band that rotates the stream, late layers that converge on logits). If the structure is uniform, the model didn't really do anything — likely the wrong dtype or a broken hook.

In [ ]:
def layer_to_layer_cosine(resid):
    """resid: (n_layers, seq_len, d_model) → (n_layers, n_layers) avg cosine across tokens."""
    normed = resid / (resid.norm(dim=-1, keepdim=True) + 1e-9)
    # einsum: for each (l1, l2) pair, average cosines over seq positions
    return torch.einsum('lsd,ksd->lks', normed, normed).mean(dim=-1).numpy().transpose()

ll_compliance = layer_to_layer_cosine(resid_compliance)
ll_refusal = layer_to_layer_cosine(resid_refusal)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, mat, title in zip(
    axes,
    [ll_compliance, ll_refusal],
    ['compliance: layer × layer cos', 'refusal: layer × layer cos'],
):
    sns.heatmap(mat, ax=ax, cmap='viridis', vmin=-0.2, vmax=1.0, cbar_kws={'shrink': 0.6})
    ax.set_title(title)
    ax.set_xlabel('layer')
    ax.set_ylabel('layer')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Checkpoint to `data/`

Phase 2 (`02_sae_features.ipynb`) needs these activations to encode through a Gemma Scope SAE. Save now to skip the model-load cost next time. The spec's data-flow note (`vault/01-projects/code/mech-interp-codebase.md` § Architecture) calls this out explicitly.

In [ ]:
payload = {
    'model_name': 'gemma-2-2b',
    'n_layers': n_layers,
    'd_model': d_model,
    'prompts': {'compliance': P_COMPLIANCE, 'refusal': P_REFUSAL},
    'token_labels': token_labels,
    'resid_pre': {
        'compliance': resid_compliance,  # (n_layers, seq_len, d_model)
        'refusal': resid_refusal,
    },
    'diagnostics': {
        'norm_compliance': torch.tensor(norm_compliance),
        'norm_refusal': torch.tensor(norm_refusal),
        'cos_matched': torch.tensor(cos_matched),
        'll_cos_compliance': torch.tensor(ll_compliance),
        'll_cos_refusal': torch.tensor(ll_refusal),
    },
}
out_path = DATA_DIR / 'phase1_residual_stream.pt'
torch.save(payload, out_path)
print(f'wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')

## 9. Observations to log

When you've run the cells, jot the answers below into `## Log` of `vault/01-projects/code/mech-interp-codebase.md` and into the Phase-1 findings file the orchestrator will append back into the linked research notes. The point is to record what the tooling *actually exposed* vs what the literature claims.

Prompts:

1. **Norm growth.** Did `‖x‖₂` grow monotonically with depth on both prompts? If not, where did it dip — and was the dip the same position on both prompts?
2. **Cross-prompt cosine onset.** At what layer does the divergent-token cosine drop below ~0.7 between the compliance and refusal prompts? Compare to Lindsey et al. 2025's mid-layer refusal-direction claim.
3. **Layer-to-layer block structure.** Are there visible 'phases' in the within-prompt layer-by-layer cosine plot? Where are the transitions?
4. **Open question forwarded to Phase 2.** Are the divergent-token cosines monosemantic-looking (a clean drop) or noisy (multiple bumps)? Clean drops are easier targets for SAE feature decomposition.

## Open issues to surface to the user

- Gemma-2-2B is base, not instruction-tuned. The refusal vs compliance contrast may surface 'this is a harm topic' features rather than 'I should refuse' features. The literature's refusal-direction work uses `-it` variants — note this gap when interpreting Diagnostic B.
- The matched-position cosine assumes positionally-aligned tokens carry comparable representations across the two prompts. The shared scaffold tokens are aligned; the slot tokens (`'bake'` vs `'make'`, `'a chocolate cake'` vs `'a lethal toxin at home'`) are *not* doing the same job. Treat divergent-token cosines as 'difference signal', not 'same-thing-different-context' signal.